# Synthetic Object Detection

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/O-2wice/synthetic-object-detection/blob/main/notebooks/object-detection.ipynb)
![Runtime](https://img.shields.io/badge/runtime-CPU%20or%20GPU-blue)
![Framework](https://img.shields.io/badge/framework-PyTorch-orange)

<img src="https://miro.medium.com/v2/resize:fit:1400/1*hrXwIJgw51P6p1xJh_IK-g.png" alt="Object detection illustration" border="0">

This project explores a controlled object detection problem: identify a character
in a cluttered scene and predict its bounding box. Each scene contains one object.
Compositing a transparent cut-out at a known location produces the image and its
class and location labels together.

The experiment develops a custom PyTorch detector with a pretrained CNN backbone,
a classification head and a bounding-box regression head. ResNet18 and VGG16 are
inspected as backbone options; the custom training run uses ResNet18. YOLOv8n is
fine-tuned on the same generated images as a reference.

The notebook follows the complete workflow: object preparation, background
collection, synthetic image generation, data loading, model inspection, training,
evaluation and visual comparison. The workflow diagram below introduces that
sequence.

In [ ]:
import os
import shutil
from pathlib import Path
if os.name == "nt" and Path("C:/Program Files/Graphviz/bin").exists():
    os.environ["PATH"] += os.pathsep + "C:/Program Files/Graphviz/bin"
import sys
import importlib.util
import subprocess
import zipfile
try:
    _colab_runtime = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    _colab_runtime = False
def _prepare_github_runtime():
    import requests
    _project = Path('/content/synthetic-object-detection')
    _archive = _project/'data/distribution/synthetic-scenes-v1.zip'
    if not (_project/'assets/objects/Waldo.png').exists() or not _archive.exists():
        _url = 'https://raw.githubusercontent.com/O-2wice/synthetic-object-detection/main/scripts/fetch_github_assets.py'
        with requests.get(_url,timeout=30) as _response:
            if _response.status_code != 200:
                raise RuntimeError(f'GitHub setup download failed (HTTP {_response.status_code}).')
            _namespace = {'__name__':'github_setup'}
            exec(compile(_response.content.decode('utf-8-sig'),'fetch_github_assets.py','exec'),_namespace)
        _namespace['fetch_project'](_project)
    os.chdir(_project)
    subprocess.check_call([sys.executable,'-m','pip','install','-q','-r','requirements.txt'])
    if shutil.which('dot') is None:
        subprocess.check_call(['apt-get','-qq','update'])
        subprocess.check_call(['apt-get','-qq','install','-y','graphviz'])

if _colab_runtime:
    _prepare_github_runtime()

from graphviz import Digraph
from IPython.display import Image as IMG

def create_flowchart(output_filename='flowchart'):
    dot = Digraph(name='Simplified Object Detection', format='png')
    dot.attr(rankdir='LR')
    dot.attr('node', shape='box', style='filled', fontsize='12', fontname='Arial')

    dot.node('Imports', 'Necessary Imports', fillcolor='#A0E7A0')

    dot.node('LoadObjects', 'Load in Your Objects', fillcolor='#A0E7A0')
    dot.node('LoadBackgrounds', 'Load in Your Backgrounds', fillcolor='#A0E7A0')

    dot.node('AugmentData', 'Define Data Augmentations', fillcolor='#A0E7A0')
    dot.node('DatasetDataloader', 'Create a Dataset and Dataloaders', fillcolor='#A0E7A0')

    dot.node('VisualizeSample', 'Visualize Training Data', fillcolor='#A0E7A0')

    dot.node('CreateModel', 'Create an Object Detection Model', fillcolor='#F6D49A')

    dot.node('TorchSummary', 'Plot Model Parameter Count and Size', fillcolor='#A0E7A0')
    dot.node('Hyperparameters', 'Define Hyperparameters', fillcolor='#A0E7A0')

    dot.node('TrainModel', 'Train the Custom Object Detection Model', fillcolor='#F6D49A')

    dot.node('VisualizeTrain', 'Visualize Training Metrics', fillcolor='#A0E7A0')
    dot.node('RunInference', 'Run Inference on the Object Detection Models', fillcolor='#F6A0A0')
    dot.node('VisPred', 'Visualize Model Predictions', fillcolor='#F6A0A0')


    dot.node('LoadYOLO', 'Load an Existing Object Detection Model', fillcolor='#F6D49A')

    dot.node('EvalYOLO', 'Evaluate the YOLO Model', fillcolor='#F6D49A')

    # Edges
    dot.edge('Imports', 'LoadObjects')
    dot.edge('Imports', 'LoadBackgrounds')

    dot.edge('LoadObjects', 'AugmentData')
    dot.edge('LoadBackgrounds', 'AugmentData')

    dot.edge('AugmentData', 'DatasetDataloader')

    dot.edge('DatasetDataloader', 'VisualizeSample')

    dot.edge('VisualizeSample', 'CreateModel')

    dot.edge('CreateModel', 'TorchSummary')
    dot.edge('CreateModel', 'Hyperparameters')

    dot.edge('Hyperparameters', 'TrainModel')

    dot.edge('TrainModel', 'VisualizeTrain')
    dot.edge('TrainModel', 'RunInference')
    dot.edge('TrainModel', 'VisPred')

    dot.edge('VisualizeSample', 'LoadYOLO')
    dot.edge('LoadYOLO', 'EvalYOLO')

    dot.render(output_filename, view=False)

create_flowchart('object_detection_flowchart')
IMG('object_detection_flowchart.png')


## Project Workflow

1. Prepare the environment and load the object cut-outs.
2. Collect and inspect cluttered background images.
3. Generate synthetic scenes and normalized bounding-box labels.
4. Build datasets and dataloaders, then inspect the training batches.
5. Define the custom detector and compare its backbone parameter counts.
6. Train with classification and box losses, validation monitoring and early stopping.
7. Plot training curves, evaluate held-out images and inspect predicted boxes.
8. Fine-tune YOLOv8n on the same images and inspect its training and evaluation results.

The task deliberately contains one object per image. It isolates classification
and localization without requiring the custom model to handle multiple detections.
Fixed cut-outs and synthetic backgrounds also limit what the results can say
about unfamiliar artwork or real scenes.

**Author:** Robert Ouko Oyombe

## 0. Environment Setup

PyTorch provides the models and training tools. Pillow and NumPy handle image
composition, while Matplotlib displays the objects, scenes, learning curves and
predictions. Training uses a GPU when one is available.

For the VS Code Colab extension, select the Colab kernel and run from the first cell. Setup downloads the saved PNGs, backgrounds and exact dataset from the public GitHub repository. No token or Drive upload is required for data loading. Google Drive is enabled for checkpoint persistence. In VS Code, use **Colab: Mount Google Drive to Server...** if prompted. See `COLAB.md` for setup and checkpoint persistence.


In [ ]:
import torch  # PyTorch for tensors and neural networks
import os    # OS operations (file paths, directories)
import numpy as np  # Numerical computations and arrays
import matplotlib.pyplot as plt  # Data visualization
import torchvision.transforms as transforms  # Image transformations
from torch import nn  # Neural network layers
from torch.utils.data import Dataset  # Custom datasets
from torch.utils.data import DataLoader  # Batch data loading
import warnings  # Warning control
# Warnings remain visible so data and runtime problems can be diagnosed.

import time  # Time functions
import torch.optim as optim  # Optimization algorithms
from tqdm import tqdm  # Progress bar
import torchvision.models as models  # Predefined models
import requests  # HTTP requests
import random  # Random number generation
from PIL import Image  # Image processing
import matplotlib.patches as patches  # Drawing shapes on plots
import shutil  # High-level file operations
import io  # I/O operations
import subprocess  # Running subprocesses
from pathlib import Path  # Path manipulations
from io import BytesIO  # Byte stream handling/ calculating model size
import cv2  # Computer vision functions
import json  # For saving results




# Set device to GPU if available, otherwise use CPU
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

# Print the device in use
print(f"Using device: {device}")
# Colab controls affect runtime and persistence, not the detector architecture.
USE_DRIVE = True  # @param {type:"boolean"}
USE_MIXED_PRECISION = True  # @param {type:"boolean"}
TRAIN_CUSTOM = True  # @param {type:"boolean"}
TRAIN_YOLO = True  # @param {type:"boolean"}
SEED = 42  # @param {type:"integer"}
# Spatial grid the neck keeps before the heads. 1 is the original global average
# pooling; 3 retains a coarse sense of position for the box head. It changes the
# shape of the head weights, so checkpoints live in a per-grid folder and a run
# started at one value cannot resume from another.
POOL_GRID = 3  # @param {type:"integer"}
import sys
import hashlib
import importlib.util
import zipfile
import gc
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p/'assets/objects/Waldo.png').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Extract the project folder (including assets) and run this notebook inside it.')
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
DRIVE_DIR = None
if IN_COLAB and USE_DRIVE:
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive')
        if not Path('/content/drive/MyDrive').is_dir():
            raise RuntimeError('Google Drive did not mount.')
        DRIVE_DIR = Path('/content/drive/MyDrive/synthetic-object-detection')
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    except Exception as exc:
        DRIVE_DIR = None
        raise RuntimeError('Mount Drive with VS Code command: Colab: Mount Google Drive to Server... Then rerun this cell. Training has not started.') from exc
WORK_DIR = PROJECT_ROOT/'data/notebook-work'
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
DATASET_DIR = WORK_DIR/'dataset'
DISTRIBUTION_DIR = PROJECT_ROOT/'data/distribution'
DATA_ARCHIVE = DISTRIBUTION_DIR/'synthetic-scenes-v1.zip'
DATA_SHA_FILE = PROJECT_ROOT/'assets/sources/dataset-archive.sha256'
OUTPUT_DIR = PROJECT_ROOT/'outputs/original-notebook'
# Every artifact that depends on the architecture is scoped by the pooling grid:
# checkpoints, metrics and figures alike. Sharing metric or figure folders would
# let a pool1 run silently overwrite pool3 results, which is the failure the
# separate checkpoint folders were meant to prevent.
RUN_TAG = f'pool{POOL_GRID}'
MODEL_DIR = ((DRIVE_DIR/'models/original-notebook') if DRIVE_DIR else OUTPUT_DIR/'models')/RUN_TAG
METRIC_DIR = ((DRIVE_DIR/'metrics') if DRIVE_DIR else OUTPUT_DIR/'metrics')/RUN_TAG
FIGURE_DIR = ((DRIVE_DIR/'figures') if DRIVE_DIR else OUTPUT_DIR/'figures')/RUN_TAG
for directory in [MODEL_DIR,METRIC_DIR,FIGURE_DIR]:
    directory.mkdir(parents=True,exist_ok=True)
BEST_PATH = MODEL_DIR/'best_model.pth'
LAST_PATH = MODEL_DIR/'last_checkpoint.pth'
YOLO_PROJECT = (DRIVE_DIR/'yolo') if DRIVE_DIR else OUTPUT_DIR/'yolo'
YOLO_RUN = YOLO_PROJECT/'train'
YOLO_LAST = YOLO_RUN/'weights/last.pt'
YOLO_BEST = YOLO_RUN/'weights/best.pt'
YOLO_ID_FILE = YOLO_PROJECT/'dataset_id.txt'
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
NUM_WORKERS = 2 if IN_COLAB else 0  # Notebook classes cannot be spawned as Windows worker imports.
USE_AMP = USE_MIXED_PRECISION and device.type == 'cuda'
print(f'Project: {PROJECT_ROOT}; runtime: {device}; mixed precision: {USE_AMP}')
print(f'Checkpoints: {MODEL_DIR}')

def sha256_file(file):
    digest = hashlib.sha256()
    with open(file,'rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def save_figure(name):
    plt.gcf().savefig(FIGURE_DIR/name,dpi=150,bbox_inches='tight')

# Record versions without collecting environment variables or credentials.
from importlib.metadata import version, PackageNotFoundError
runtime_versions = {'python':sys.version.split()[0], 'cuda':torch.version.cuda}
for package in ['torch','torchvision','ultralytics','numpy','Pillow','matplotlib','pandas']:
    try:
        runtime_versions[package] = version(package)
    except PackageNotFoundError:
        runtime_versions[package] = 'not installed'
(METRIC_DIR/'runtime.json').write_text(json.dumps(runtime_versions,indent=2),encoding='utf-8')

def export_run_artifacts():
    """Export available progress, including before the YOLO section has run."""
    bundle = OUTPUT_DIR/'run-artifacts.zip'
    temporary = bundle.with_suffix('.tmp')
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_STORED) as archive:
        for prefix,folder in [('metrics',METRIC_DIR),('figures',FIGURE_DIR)]:
            for file in sorted(folder.rglob('*')):
                if file.is_file(): archive.write(file,prefix+'/'+file.relative_to(folder).as_posix())
        for file in [BEST_PATH,LAST_PATH]:
            if file.exists(): archive.write(file,'models/'+file.name)
        if YOLO_PROJECT.exists():
            for file in sorted(YOLO_PROJECT.rglob('*')):
                if file.is_file() and file.suffix not in {'.tmp','.download'}:
                    archive.write(file,'yolo/'+file.relative_to(YOLO_PROJECT).as_posix())
        manifest = DATASET_DIR/'manifest.json'
        if manifest.exists(): archive.write(manifest,'dataset-manifest.json')
    temporary.replace(bundle)
    if DRIVE_DIR:
        destination = DRIVE_DIR/bundle.name
        pending = destination.with_suffix('.tmp')
        shutil.copy2(bundle,pending)
        pending.replace(destination)
        print(f'Run archive saved to Drive: {destination}')
    else:
        print(f'Download before ending the runtime: {bundle}')
    return bundle


### Model summary tools


In [ ]:
from torchsummary import summary


### Background crawler


In [ ]:
from icrawler.builtin import GoogleImageCrawler


### Optional background removal


In [ ]:
# Background removal is only needed for NEW objects; saved PNGs already have alpha.
# For new object downloads, install rembg and onnxruntime in this kernel first.


## 1.1 Object Loading Process

The detector learns three character classes. Transparent cut-outs make it possible
to place each character onto different backgrounds while retaining an exact
bounding box. The original character references came from the
[Where's Wally character collection](https://waldo.fandom.com/wiki/Category:Characters).

<img src="https://kotaksuratriza.wordpress.com/wp-content/uploads/2012/06/wally-and-friends.jpg" alt="Where's Wally characters" />

The object-loading cell contains the download, background-removal and display
steps. The side-by-side figure provides a visual check of the chosen objects and
their transparent boundaries before scene generation.

The original Drive links are no longer available. The approved replacement assets
are Waldo, Wenda and Wizard Whitebeard, saved with their publisher source and
preparation details in the project. Wenda replaces the original Wilma class.

In [ ]:
# Define objects (name and URL)
OBJECTS = {0: {'name': 'Waldo', 'url': 'https://raw.githubusercontent.com/O-2wice/synthetic-object-detection/main/assets/objects/Waldo.png'}, 1: {'name': 'Wenda', 'url': 'https://raw.githubusercontent.com/O-2wice/synthetic-object-detection/main/assets/objects/Wenda.png'}, 2: {'name': 'Wizard Whitebeard', 'url': 'https://raw.githubusercontent.com/O-2wice/synthetic-object-detection/main/assets/objects/Wizard%20Whitebeard.png'}}
OBJECTS_DIR = "objects"
os.makedirs(OBJECTS_DIR, exist_ok=True)  # Create directory for objects


def get_direct_download_url(gdrive_url): #Convert a Google Drive share link into a direct download link.

    file_id = gdrive_url.split("/")[5]  # Extract the file ID from the URL
    return f"https://drive.google.com/uc?id={file_id}&export=download"


def download_image(url): #Download an image from a URL, including handling Google Drive links
    try:
        # Check if the URL is a Google Drive link
        if "drive.google.com" in url:
            url = get_direct_download_url(url)

        # Add headers to mimic a browser request
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, stream=True, timeout=30)
        response.raise_for_status()

        # Check if the content type is an image
        content_type = response.headers.get('content-type')
        if not content_type or not content_type.startswith('image'):
            print(f"Error: URL does not point to an image. Content type: {content_type}")
            return None

        return response.content
    except Exception as e:
        print(f"Error downloading image: {e}")
        return None


def remove_background(image_bytes):
    """Remove background from an image using rembg."""
    try:
        img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        from rembg import remove
        result = remove(np.array(img))  # Remove background
        return Image.fromarray(result)
    except Exception as e:
        print(f"Error removing background: {e}")
        return None


def save_image(image, filename):
    """Save an image to a file."""
    try:
        image.save(filename, "PNG")
    except Exception as e:
        print(f"Error saving image: {e}")


def process_objects():
    """Download, process, and save objects with backgrounds removed."""
    for obj_data in OBJECTS.values():
        name = obj_data["name"]
        url = obj_data["url"]
        removed_bg_path = os.path.join(OBJECTS_DIR, f"{name}.png")  # Save only the processed image

        # Skip if already processed
        if os.path.exists(removed_bg_path):
            print(f"Skipping {name} (already processed)")
            continue

        # Download image
        img_bytes = download_image(url)
        if not img_bytes:
            print(f"Skipping {name} (download failed)")
            continue

        # Remove background and save
        removed_bg_img = remove_background(img_bytes)
        if removed_bg_img:
            save_image(removed_bg_img, removed_bg_path)
            print(f"Processed and saved {name}")
        else:
            print(f"Skipping {name} (background removal failed)")


def display_objects():
    """Display all processed objects side by side."""
    plt.figure(figsize=(12, 4))  # Reduced figure size for better visualization

    for i, obj_data in enumerate(OBJECTS.values()):
        name = obj_data["name"]
        removed_bg_path = os.path.join(OBJECTS_DIR, f"{name}.png")

        if os.path.exists(removed_bg_path):
            removed_bg_img = Image.open(removed_bg_path)

            # Display the processed image
            plt.subplot(1, len(OBJECTS), i + 1)  # Arrange images in a single row
            plt.title(name)  # Title with the object's name
            plt.imshow(removed_bg_img)
            plt.axis('off')  # Turn off axis labels

    plt.tight_layout()  # Adjust layout to prevent overlap
    save_figure('objects.png')
    plt.show()


# Reuse PNGs fetched from GitHub by setup. Original Drive links remain in DATA.md.
OBJECTS[1]['name'] = 'Wenda'  # Approved replacement for the unavailable Wilma image.
for info in OBJECTS.values():
    source = PROJECT_ROOT/'assets/objects'/f"{info['name']}.png"
    if not source.exists():
        raise FileNotFoundError(f'Missing saved object: {source}')
    shutil.copy2(source,Path(OBJECTS_DIR)/source.name)
# process_objects()  # Use only when deliberately preparing new source images.
display_objects()


## 1.2 Crawling the Web for Background Images


The synthetic scenes use cluttered doodle backgrounds. The crawling step collects
candidate images, and the following visualization makes the collection inspectable
before objects are placed into it.

For repeatable runs, the saved background collection is reused. A new crawl is a
new source collection, because search results and remote files can change.
[icrawler documentation](https://icrawler.readthedocs.io/en/latest/) describes the
crawler used in this step.

In [ ]:
Path('background').mkdir(exist_ok=True)
review = json.loads((PROJECT_ROOT/'assets/backgrounds/review.json').read_text())
excluded_backgrounds = set(review['excluded'])
for source in sorted((PROJECT_ROOT/'assets/backgrounds').glob('*.jpg')):
    if source.name not in excluded_backgrounds:
        shutil.copy2(source,Path('background')/source.name)


Google, Bing and Baidu crawlers are alternatives for collecting a new background
set. The experiment uses its saved collection once that collection has been
reviewed; crawling is a data-preparation step rather than a requirement on every run.

In [ ]:
CRAWL_NEW_BACKGROUNDS = False  # @param {type:"boolean"}
if CRAWL_NEW_BACKGROUNDS:
    google_crawler = GoogleImageCrawler(
        parser_threads=2,
        downloader_threads=4,
        storage={'root_dir': 'background'}
    )
    
    for keyword in ['doodle background', 'cluttered doodle background', 'colorful doodle backgroun', 'waldo doodle background']:
        google_crawler.crawl(
            keyword=keyword, max_num=100, file_idx_offset='auto')
    
    # Some images may return an error, but icrawler tries to find other images regardless


### Inspecting the backgrounds

A small sample shows the variety of colors, textures and clutter available for
scene generation. Backgrounds should not already contain a target character,
since each generated scene is labeled as containing exactly one object.

In [ ]:
#  Visualizing a small subset of the background images

def display_sample_images(directory, num_samples=5):  #Displays a small subset of images from a given directory.
    try:
        # Get all image files in the directory (supports .png, .jpg, .jpeg)
        image_files = [f for f in os.listdir(directory) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        # Check if there are any images in the directory
        if not image_files:
            print(f"No image files found in directory: {directory}")
            return

        # Randomly select a subset of images (up to `num_samples`)
        sample_files = random.sample(image_files, min(num_samples, len(image_files)))

        # Display images in a grid
        plt.figure(figsize=(15, 5))  # Set figure size
        for i, file_name in enumerate(sample_files):
            image_path = os.path.join(directory, file_name)  # Full path to the image
            img = Image.open(image_path)  # Open the image

            # Display the image with its file name as the title
            plt.subplot(1, len(sample_files), i + 1)  # Arrange images in a single row
            plt.imshow(img)
            plt.title(file_name, fontsize=10)  # Show file name as title
            plt.axis('off')  # Turn off axis labels

        plt.tight_layout()  # Adjust layout to prevent overlap
        save_figure('backgrounds.png')
        plt.show()

    except FileNotFoundError:
        print(f"Directory not found: {directory}")  # Handle missing directory
    except Exception as e:
        print(f"An error occurred: {e}")  # Handle other exceptions

# Display a small subset of images from the 'background' directory
display_sample_images("background")


## 2. Creating a Synthetic Dataset

The generator selects a background and one object, places the object at a random
valid position, and saves the composite with its label. If an object of width
$w$ and height $h$ is placed at $(x,y)$ in an image of width $W$ and height $H$,
its normalized box is

$$
[x_{\mathrm{center}},y_{\mathrm{center}},\mathrm{width},\mathrm{height}]
=\left[\frac{x+w/2}{W},\frac{y+h/2}{H},\frac{w}{W},\frac{h}{H}\right].
$$

Each label row stores `<class_id> <x_center> <y_center> <width> <height>`.
This YOLO format supports both the custom detector and the YOLO reference.

Images are generated at 640 × 640 pixels, with 5,000 training, 1,000 validation
and 200 test examples. The original `createDataset` function defines how scenes were generated. Normal
local and Colab runs download the same frozen archive rather than regenerate it.
`ObjectDetectionDataset` loads the resulting pairs for PyTorch.

The original generator samples backgrounds from a common source pool. The test
set therefore measures new placements and composites, not unseen-background
generalization. That distinction remains part of the experiment's scope.

<img src="https://i.ibb.co/XrHqLmz4/2.png" alt="Synthetic object placement and bounding-box illustration" border="0">

In [ ]:
# Define objects (name and path)
OBJECTS_DIR = "objects"
OBJECTS = {
    0: {"name": "Waldo", "path": os.path.join(OBJECTS_DIR, "Waldo.png")},
    1: {"name": "Wenda", "path": os.path.join(OBJECTS_DIR, "Wenda.png")},
    2: {"name": "Wizard Whitebeard", "path": os.path.join(OBJECTS_DIR, "Wizard Whitebeard.png")},
}

BACKGROUND_DIR = "background"

def createDataset(root_dir, dataset_name, split, img_size, num_images):
    """
    Creates a synthetic dataset by placing objects onto background images.

    Args:
        root_dir (str): The root directory for the dataset.
        dataset_name (str): The name of the dataset.
        split (str): The dataset split (e.g., 'train', 'val', 'test').
        img_size (tuple): The desired size of the output images (width, height).
        num_images (int): The number of images to generate for the split.
    """
    # Ensure the root directory and all necessary subdirectories exist
    dataset_path = os.path.join(root_dir, dataset_name, split)
    images_path = os.path.join(dataset_path, "images")
    labels_path = os.path.join(dataset_path, "labels")

    # Create the directories if they don't exist
    os.makedirs(images_path, exist_ok=True)
    os.makedirs(labels_path, exist_ok=True)

    # Ensure the background files are present
    background_files = [f for f in sorted(os.listdir(BACKGROUND_DIR)) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if not background_files:
        raise ValueError('No backgrounds are available.')
    for filename in background_files:
        with Image.open(os.path.join(BACKGROUND_DIR,filename)) as check:
            check.verify()
    for obj in OBJECTS.values():
        with Image.open(obj['path']) as check:
            bounds = check.convert('RGBA').getchannel('A').getbbox()
            if bounds is None:
                raise ValueError(f"Empty cut-out: {obj['name']}")
            if bounds[2]-bounds[0] > img_size[0]-10 or bounds[3]-bounds[1] > img_size[1]-10:
                raise ValueError(f"Object does not fit: {obj['name']}")
    # Each split has an independent seed, so restarting after a completed split
    # does not change the subsequent placements.
    random.seed(SEED + {'train':0,'val':1,'test':2}[split])
    created_images = 0
    while created_images < num_images:
        background_file = random.choice(background_files)
        background_path = os.path.join(BACKGROUND_DIR, background_file)
        try:
            background_img = Image.open(background_path).convert("RGB").resize(img_size)
        except Exception as e:
            print(f"Error opening background image {background_path}: {e}. Skipping.")
            continue

        background_np = np.array(background_img)
        img_height, img_width, _ = background_np.shape

        # Randomly select an object
        object_id = random.choice(list(OBJECTS.keys()))
        object_path = OBJECTS[object_id]["path"]
        try:
            object_img = Image.open(object_path).convert("RGBA")
        except Exception as e:
            print(f"Error opening object image {object_path}: {e}. Skipping.")
            continue

        object_np = np.array(object_img)

        # Get object dimensions (handling transparent images) and crop
        alpha_channel = object_np[:, :, 3] if object_np.shape[2] == 4 else np.ones_like(object_np[:, :, 0])
        non_transparent_pixels = np.argwhere(alpha_channel > 0)
        if non_transparent_pixels.size > 0:
            min_y, min_x = non_transparent_pixels.min(axis=0)[:2]
            max_y, max_x = non_transparent_pixels.max(axis=0)[:2]
            object_height = max_y - min_y + 1
            object_width = max_x - min_x + 1

            # Crop object image to remove extra transparent space
            object_np = object_np[min_y:max_y + 1, min_x:max_x + 1]
        else:
            object_height, object_width = 0, 0

        # Ensure object fits within the background
        max_x_pos = img_width - object_width
        max_y_pos = img_height - object_height

        # Add a small buffer to prevent objects from being placed too close to the edge
        buffer = 5  # Adjust buffer as needed
        x_pos = random.randint(buffer, max_x_pos - buffer) if max_x_pos - buffer > buffer else 0
        y_pos = random.randint(buffer, max_y_pos - buffer) if max_y_pos - buffer > buffer else 0

        # Check if the object fits within the background
        if x_pos + object_width > img_width or y_pos + object_height > img_height:
            continue  # Skip this image if the object doesn't fit

        # Create synthetic image
        if object_width > 0 and object_height > 0:
            synthetic_img = background_np.copy()
            for c in range(0, 3):
                alpha = object_np[:, :, 3] / 255.0
                region = synthetic_img[y_pos:y_pos + object_height, x_pos:x_pos + object_width, c]
                synthetic_img[y_pos:y_pos + object_height, x_pos:x_pos + object_width, c] = (
                    alpha*object_np[:,:,c] + (1-alpha)*region).round().astype(np.uint8)
        else:
            synthetic_img = background_np.copy()

        synthetic_img = Image.fromarray(synthetic_img)
        synthetic_img.save(os.path.join(images_path, f"{created_images:05d}.jpg"))

        # Create YOLO label
        if object_width > 0 and object_height > 0:
            x_center = (x_pos + object_width / 2) / img_width
            y_center = (y_pos + object_height / 2) / img_height
            width = object_width / img_width
            height = object_height / img_height

            with open(os.path.join(labels_path, f"{created_images:05d}.txt"), "w") as f:
                f.write(f"{object_id} {x_center} {y_center} {width} {height}")
        else:
            with open(os.path.join(labels_path, f"{created_images:05d}.txt"), "w") as f:
                f.write('')

        created_images += 1

    print(f"Dataset {split} created with {num_images} images.")

# One frozen local dataset is used by both models and copied unchanged to Colab.
def verify_dataset():
    manifest_path = DATASET_DIR/'manifest.json'
    reference = PROJECT_ROOT/'assets/sources/dataset-manifest.json'
    if not reference.exists() or json.loads(manifest_path.read_text()) != json.loads(reference.read_text()):
        raise ValueError('Dataset manifest differs from the saved project reference.')
    manifest = json.loads(manifest_path.read_text())
    if manifest['seed'] != SEED:
        raise ValueError('Dataset seed differs. Use its saved seed or create a separately named dataset.')
    for relative, expected in manifest['files'].items():
        if sha256_file(DATASET_DIR/relative) != expected:
            raise ValueError(f'Dataset content changed: {relative}')
    expected_files = set(manifest['files'])
    actual_files = {p.relative_to(DATASET_DIR).as_posix() for p in (DATASET_DIR/'background').rglob('*') if p.is_file() and p.parent.name in {'images','labels'}}
    if actual_files != expected_files:
        raise ValueError('Dataset contains missing or extra files.')
    return sha256_file(manifest_path)

def prepare_fixed_dataset():
    if not (DATASET_DIR/'manifest.json').exists():
        if not DATA_ARCHIVE.exists() and DRIVE_DIR and (DRIVE_DIR/'data'/DATA_ARCHIVE.name).exists():
            DISTRIBUTION_DIR.mkdir(parents=True,exist_ok=True)
            shutil.copy2(DRIVE_DIR/'data'/DATA_ARCHIVE.name,DATA_ARCHIVE)
        if DATA_ARCHIVE.exists():
            if not DATA_SHA_FILE.exists() or sha256_file(DATA_ARCHIVE) != DATA_SHA_FILE.read_text().strip():
                raise ValueError('Dataset archive hash mismatch.')
            with zipfile.ZipFile(DATA_ARCHIVE) as archive:
                for member in archive.infolist():
                    target = (DATASET_DIR/member.filename).resolve()
                    if not target.is_relative_to(DATASET_DIR.resolve()):
                        raise ValueError('Invalid archive path.')
                archive.extractall(DATASET_DIR)
        else:
            import runpy
            fetch = runpy.run_path(str(PROJECT_ROOT/'scripts/fetch_github_assets.py'))['fetch_project']
            fetch(PROJECT_ROOT)
            return prepare_fixed_dataset()
    return verify_dataset()

DATASET_ID = prepare_fixed_dataset()
print(f'Frozen dataset verified: {DATASET_ID}')


### Dataset and paired augmentations


In [ ]:
def augment_image_and_boxes(image, labels):
    """Original horizontal flip and +/-10 degree rotation, now also applied to boxes."""
    labels = labels.copy()
    if random.random() < .5:
        image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        labels[:,1] = 1-labels[:,1]
    angle = random.uniform(-10,10)
    width,height = image.size
    image = transforms.functional.rotate(image,angle)
    theta = np.deg2rad(angle)
    for row in labels:
        _,cx,cy,w,h = row
        corners = np.array([[(cx-w/2)*width,(cy-h/2)*height],
                            [(cx+w/2)*width,(cy-h/2)*height],
                            [(cx+w/2)*width,(cy+h/2)*height],
                            [(cx-w/2)*width,(cy+h/2)*height]])
        center = np.array([width/2,height/2])
        matrix = np.array([[np.cos(theta),np.sin(theta)],[-np.sin(theta),np.cos(theta)]])
        rotated = (corners-center)@matrix.T+center
        low = np.maximum(rotated.min(axis=0),[0,0])
        high = np.minimum(rotated.max(axis=0),[width,height])
        row[1:] = [*( (low+high)/2/[width,height] ),*( (high-low)/[width,height] )]
    return image,labels

class ObjectDetectionDataset(Dataset):
    """
    Dataset class for loading object detection data.
    """
    def __init__(self, root_dir, split, num_classes, transform=None, augment=False):
        """
        Initializes the dataset.

        Args:
            root_dir (str): The root directory of the dataset.
            split (str): The dataset split (e.g., 'train', 'val', 'test').
            num_classes (int): The number of object classes.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.root_dir = root_dir
        self.split = split
        self.images_dir = os.path.join(root_dir, "background", split, "images")
        self.labels_dir = os.path.join(root_dir, "background", split, "labels")
        self.image_files = sorted(os.listdir(self.images_dir))
        self.num_classes = num_classes
        self.transform = transform
        self.augment = augment

    def __len__(self):
        """
        Returns the size of the dataset.
        """
        return len(self.image_files)

    def __getitem__(self, idx):
        """
        Gets a sample from the dataset.

        Args:
            idx (int): Index of the sample.

        Returns:
            tuple: (image, labels) where labels is a tensor containing bounding box coordinates.
        """
        image_path = os.path.join(self.images_dir, self.image_files[idx])
        label_path = os.path.join(self.labels_dir, self.image_files[idx].replace(".jpg", ".txt"))

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"Error opening image {image_path}: {e}")
            raise

        # Initialize an empty list to store all the bounding boxes
        labels = []

        # Read the label file
        try:
            with open(label_path, 'r') as file:
                for line in file:
                    if line.strip():  # Avoid empty lines
                        class_id, x_center, y_center, width, height = map(float, line.strip().split())
                        labels.append([class_id, x_center, y_center, width, height])
        except FileNotFoundError:
            pass  # If there's no label file, just return an empty list

        # Convert labels to numpy array (if any objects exist)
        if len(labels) > 0:
            labels = np.array(labels)
        else:
            labels = np.array([[0, 0, 0, 0, 0]])  # Default empty label if no object exists

        if labels.shape != (1,5) or not np.isfinite(labels).all():
            raise ValueError(f'Expected one valid target in {label_path}')
        cls,cx,cy,w,h = labels[0]
        if cls != int(cls) or not 0 <= cls < self.num_classes or min(w,h) <= 0:
            raise ValueError(f'Invalid label in {label_path}')
        if min(cx-w/2,cy-h/2) < -1e-6 or max(cx+w/2,cy+h/2) > 1+1e-6:
            raise ValueError(f'Out-of-bounds label in {label_path}')
        if self.augment:
            image, labels = augment_image_and_boxes(image,labels)
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(labels, dtype=torch.float32)

# Define transforms
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Create dataset instances
train_dataset = ObjectDetectionDataset(root_dir='dataset', split='train', num_classes=3, transform=train_transform, augment=True)
val_dataset = ObjectDetectionDataset(root_dir='dataset', split='val', num_classes=3, transform=val_transform)
test_dataset = ObjectDetectionDataset(root_dir='dataset', split='test', num_classes=3, transform=test_transform)


### Inspect the generated scenes


In [ ]:
# Visualizing dataset (Optional)

def visualize_dataset(root_dir, dataset_name, split, num_samples=8): # Visualizes images from the generated dataset with their bounding boxes.

    images_dir = os.path.join(root_dir, dataset_name, split, "images")
    labels_dir = os.path.join(root_dir, dataset_name, split, "labels")
    image_files = sorted(os.listdir(images_dir))

    if not image_files:
        print(f"No images found in {images_dir}")
        return

    selected_files = random.sample(image_files, min(num_samples, len(image_files)))

    plt.figure(figsize=(16, 10))
    for i, image_file in enumerate(selected_files):
        image_path = os.path.join(images_dir, image_file)
        label_file = os.path.join(labels_dir, image_file.replace(".jpg", ".txt"))

        try:
            image = Image.open(image_path).convert("RGB")
            image_np = np.array(image)
            img_height, img_width, _ = image_np.shape

            with open(label_file, "r") as f:
                label_data = f.readline().strip()
            if label_data:
                label = list(map(float, label_data.split()))
                class_id, x_center, y_center, width, height = label

                x = (x_center - width / 2) * img_width
                y = (y_center - height / 2) * img_height
                w = width * img_width
                h = height * img_height

                plt.subplot(2, 4, i + 1)
                plt.imshow(image_np)
                rect = patches.Rectangle((x, y), w, h, linewidth=1, edgecolor='r', facecolor='none')
                plt.gca().add_patch(rect)
                plt.title(f"Class: {int(class_id)}")
                plt.axis('off')
            else:
                plt.subplot(2, 4, i + 1)
                plt.imshow(image_np)
                plt.title(f"No Object")
                plt.axis('off')

        except FileNotFoundError:
            print(f"Error: Image or label file not found for {image_file}")
        except ValueError:
            print(f"Error: Invalid label format in {label_file}")
        except Exception as e:
            print(f"An error occurred during visualization: {e}")

    plt.tight_layout()
    save_figure(f'dataset-{split}.png')
    plt.show()

# Visualization:
root_dir = 'dataset'
dataset_name = 'background'

visualize_dataset(root_dir, dataset_name, split='train', num_samples=8)
visualize_dataset(root_dir, dataset_name, split='val', num_samples=8)
visualize_dataset(root_dir, dataset_name, split='test', num_samples=8)


### Save the dataset to Drive


In [ ]:
def save_dataset_to_drive(local_dataset_path, drive_dataset_path):
    if DRIVE_DIR is None:
        print('Drive is not mounted; the local dataset archive remains available.')
        return
    destination = DRIVE_DIR/'data'/DATA_ARCHIVE.name
    destination.parent.mkdir(parents=True,exist_ok=True)
    if destination.exists() and sha256_file(destination) == DATA_SHA_FILE.read_text().strip():
        print('Identical dataset archive already backed up.')
        return
    if destination.exists():
        raise FileExistsError('A different Drive dataset exists; choose a new destination.')
    shutil.copy2(DATA_ARCHIVE,destination)
    print(f'Fixed dataset copied to {destination}')

save_dataset_to_drive(DATASET_DIR,'data')


### Verify the restored dataset


In [ ]:
def retrieve_dataset_from_drive(drive_dataset_path, local_dataset_path):
    if (DATASET_DIR/'manifest.json').exists():
        print(f'Local dataset verified: {verify_dataset()}')
        return
    if DRIVE_DIR is None:
        raise FileNotFoundError('Copy the local dataset archive to this runtime first.')
    DISTRIBUTION_DIR.mkdir(parents=True,exist_ok=True)
    shutil.copy2(DRIVE_DIR/'data'/DATA_ARCHIVE.name,DATA_ARCHIVE)
    prepare_fixed_dataset()

retrieve_dataset_from_drive('data',DATASET_DIR)


## 3. Creating Dataloaders

Dataloaders batch the generated images and labels. Training examples are shuffled;
validation and test loaders support repeatable evaluation. The batch-size setting
controls memory use, and the diagnostic output shows the number of batches in
each split.

In [ ]:
# Define batch size based on available memory (adjust as needed)
batch_size = 16  # ** system's memory capacity

# Function to create DataLoader with memory efficiency considerations

def create_dataloader(dataset, batch_size, shuffle, num_workers=NUM_WORKERS, pin_memory=True):  #Creates a DataLoader with memory efficiency considerations.

#DataLoader: The DataLoader instance.

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,

        pin_memory=pin_memory and device.type == "cuda"
    )
    return dataloader

# Create DataLoader instances
train_loader = create_dataloader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = create_dataloader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = create_dataloader(test_dataset, batch_size=batch_size, shuffle=False)

# Optional: Print the size of batches to verify everything
print(f"Training batch size: {batch_size}, Number of batches: {len(train_loader)}")
print(f"Validation batch size: {batch_size}, Number of batches: {len(val_loader)}")
print(f"Test batch size: {batch_size}, Number of batches: {len(test_loader)}")


## 4. Visualizing the Training Data

The next figure displays a training batch with class labels and bounding boxes.
The 2 × 4 layout makes it easy to check placement, scale and label alignment
before training. Images are denormalized for display, and out-of-bounds boxes
are flagged by the visualization.

In [ ]:
#Visualizing Train Data

def visualize_batch(dataloader, num_images=8):
    images, targets = next(iter(dataloader))
    images = images[:num_images]
    targets = targets[:num_images]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for ax in axes:
        ax.axis("off")
    for i in range(min(num_images,len(images))):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)

        ax = axes[i]
        ax.imshow(img)

        if isinstance(targets, dict):
            labels = targets[i]['labels'].cpu().numpy()
            bboxes = targets[i]['boxes'].cpu().numpy()
        else:
            labels = targets[i][:, 0].cpu().numpy()
            bboxes = targets[i][:, 1:].cpu().numpy()

        for j in range(len(labels)):
            obj_id = labels[j]
            bbox = bboxes[j]

            x_center, y_center, width, height = bbox
            img_height, img_width, _ = img.shape
            x = (x_center - width / 2) * img_width
            y = (y_center - height / 2) * img_height
            w = width * img_width
            h = height * img_height

            # Enhanced Boundary Checks and Debugging
            if x < 0 or y < 0 or (x + w) > img_width or (y + h) > img_height:
                print(f"ERROR: Bounding box for object {obj_id} in image {i} is OUTSIDE image boundaries.")
                print(f"  Bbox: x={x}, y={y}, w={w}, h={h}")
                print(f"  Image shape: {img_width}, {img_height}")
                print(f"  Original bbox (center, w, h): {x_center}, {y_center}, {width}, {height}")
                print(f"  Labels: {labels}")
                print(f"  Bboxes: {bboxes}")

                # Add a red marker at out-of-bounds coordinates
                ax.plot(x, y, 'rx', markersize=10)
                ax.plot(x + w, y + h, 'rx', markersize=10)

            else:
                rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none')
                ax.add_patch(rect)
                ax.text(x, y - 5, f"{int(obj_id)}", color='r', fontsize=12, bbox=dict(facecolor='white', alpha=0.5))

        ax.axis("off")

    plt.tight_layout()
    save_figure('training-batch.png')
    plt.show()

visualize_batch(train_loader)


## 5. Creating a Custom Object Detection Model

The custom CNN detector has a pretrained feature-extraction backbone and two
output branches:

- a classification head with one logit per class;
- a regression head with four normalized box coordinates:
  `[x_center, y_center, width, height]`.

The architecture supports ResNet18 and VGG16 backbones. The pooled features feed
linear heads that use ReLU, batch normalization and dropout in the classification
branch. The backbone can be frozen or fine-tuned; the training run below
fine-tunes ResNet18.

**On the pooling.** The first version of this detector pooled the feature map to
`1 x 1` before both heads. That suits the classification head, which only has to
report what is present. It is a poor fit for the box head: averaging over every
spatial position discards the arrangement of the feature map, leaving position
to be inferred from whatever survives in the channel averages and from border
effects. Some signal does survive, so this is a weakened cue rather than an
absent one, but it is a strange thing to ask a linear layer to recover.

Pooling to a `3 x 3` grid keeps a coarse sense of position while the heads still
read a fixed-length vector, costing about two million parameters and leaving the
backbone untouched.

`POOL_GRID` controls this, and it is set with the other run settings in section
0 because the checkpoint folder depends on it. Setting it to `1` reproduces the
original global average pooling, so the two can be compared on identical data;
each grid writes to its own folder under `models/`, so the variants do not
overwrite one another. The grid is also recorded inside the checkpoint, since
changing it changes the shape of the head weights and a checkpoint written at one
setting cannot be resumed at another.

Whether the change helps is an empirical question, and the run below answers it
rather than the paragraph above.

The unconstrained box output is left as it is: the head can emit coordinates
outside `[0, 1]`, which the loss penalizes rather than prevents.

In [ ]:
class CustomObjectDetectionModel(nn.Module):
    def __init__(self, num_classes=3, backbone_type="resnet18", fine_tune_backbone=False,
                 pool_grid=POOL_GRID):
        super(CustomObjectDetectionModel, self).__init__()

        # Select backbone (ResNet or VGG)
        if backbone_type == "resnet18":
            self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            feature_dim = 512  # ResNet18 outputs 512 channels
            self.backbone = nn.Sequential(*list(self.backbone.children())[:-2])  # Remove avgpool & fc layers
        elif backbone_type == "vgg16":
            self.backbone = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
            feature_dim = 512  # VGG16 outputs 512 channels
            self.backbone = nn.Sequential(*list(self.backbone.features.children())[:-1])  # Keep feature extractor only
        else:
            raise ValueError("Unsupported backbone type. Choose 'resnet18' or 'vgg16'.")

        # Freeze backbone layers if fine_tune_backbone is False
        if not fine_tune_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Adaptive pooling to reduce feature map size.
        #
        # The original used (1, 1), which averages the feature map over every
        # spatial position. That is reasonable for the classification head,
        # which only needs to know what is present, but it discards the
        # information the box head exists to use: after a global average, where
        # the object sat is gone. Pooling to a small grid keeps a coarse sense
        # of position at negligible cost, since the heads still read a fixed
        # vector. Set POOL_GRID = 1 to reproduce the original behaviour.
        self.pool_grid = pool_grid
        self.global_pool = nn.AdaptiveAvgPool2d((pool_grid, pool_grid))
        feature_dim = feature_dim * pool_grid * pool_grid

        # Classification head: emits one unnormalized score (logit) per class.
        # CrossEntropyLoss applies the softmax, so no activation belongs here.
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)  # One logit per class
        )

        # Bounding box regression head: Predicts 4 values [x_center, y_center, width, height] for each object
        self.regressor = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 4)  # 4 values for bounding box [x_center, y_center, width, height]
        )

    def forward(self, x):
        """
        Forward pass of the model.

        Parameters:
        - x (Tensor): Input image tensor (batch_size, 3, H, W)

        Returns:
        - class_probs (Tensor): Predicted class logits (batch_size, num_classes)
        - bbox_coords (Tensor): Predicted bounding box coordinates (batch_size, 4)
        """
        # Extract features using the backbone
        features = self.backbone(x)  # (batch_size, 512, H', W')

        # Pool to the configured grid, which reduces the spatial dimensions
        # without discarding position entirely when POOL_GRID > 1.
        pooled_features = self.global_pool(features)  # (batch_size, 512, G, G)

        # Flatten features to pass through fully connected layers
        flattened_features = pooled_features.view(pooled_features.size(0), -1)  # (batch_size, 512*G*G)

        # Class probabilities: Predicting the object class for each bounding box
        class_probs = self.classifier(flattened_features)  # (batch_size, num_classes)

        # Bounding box regression: Predicting the bounding box coordinates for each object
        bbox_coords = self.regressor(flattened_features)  # (batch_size, 4)

        return class_probs, bbox_coords


# Model Instances
# Instantiate the model with ResNet18 backbone
model_resnet = CustomObjectDetectionModel(num_classes=3, backbone_type="resnet18", fine_tune_backbone=True)

# Instantiate the model with VGG16 backbone
model_vgg = CustomObjectDetectionModel(num_classes=3, backbone_type="vgg16", fine_tune_backbone=False)

# Print model details
print("ResNet Model:\n", model_resnet)
print("\nVGG Model:\n", model_vgg)


## 6.1 Plotting Model Parameter Count and Size

Model summaries show layer shapes and parameter counts for the ResNet18 and VGG16
variants. The following bar chart compares their trainable parameter counts on a
logarithmic scale. Both variants use the same custom heads.

In [ ]:
from torchvision import models
# #Plot for Unified Model

# Use the same 640x640 input dimensions as the experiment.
input_size = (3, 640, 640)  # (Channels, Height, Width)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load both models
model_resnet = CustomObjectDetectionModel(num_classes=3, backbone_type="resnet18", fine_tune_backbone=True).to(device)
model_vgg = CustomObjectDetectionModel(num_classes=3, backbone_type="vgg16", fine_tune_backbone=True).to(device)

# Print model summaries
print("\nResNet Model Summary:")
with torch.no_grad():
    summary(model_resnet, input_size=input_size, device=device.type)

print("\nVGG Model Summary:")
with torch.no_grad():
    summary(model_vgg, input_size=input_size, device=device.type)

# Function to count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Get parameter counts
resnet_params = count_parameters(model_resnet)
vgg_params = count_parameters(model_vgg)

# Plot parameter comparison
backbone_names = ["ResNet18", "VGG16"]
params = [resnet_params, vgg_params]

plt.figure(figsize=(8, 5))
plt.bar(backbone_names, params, color=['blue', 'red'])
plt.ylabel("Trainable Parameters")
plt.title("Model Parameter Count Comparison")
plt.yscale("log")  # Use log scale for better visualization
save_figure('model-parameters.png')
plt.show()

# Summary models are not needed during training.
del model_resnet,model_vgg
gc.collect()
if device.type == "cuda": torch.cuda.empty_cache()


## 6.2 Defining Loss Function and Optimizer

**Loss Functions and their options:**

For our custom object detection task, we are performing both **regression** (for bounding box coordinates) and **classification** (for object categories). Our network predicts **continuous** bounding box values, along with a **discrete** class label, meaning that our chosen loss function should effectively be a composite of two losses:  

1. **A regression loss** for bounding box prediction.  
2. **A classification loss** for object label prediction.  

Here are some common loss functions that can be used:  

---

### a. Mean Squared Error (MSE) Loss (Bounding Box Regression)
MSE loss is a standard choice for regression tasks, as it penalizes larger errors more strongly than smaller ones. For bounding box prediction, this ensures that predicted box coordinates are as close as possible to the ground truth.  

$$
\mathcal{L}_{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2
$$  

where $ y_i $ is the ground truth bounding box coordinate and $ \hat{y}_i $ is the predicted coordinate.  

- **Pros:** Simple, computationally efficient, commonly used in regression tasks.  
- **Cons:** Sensitive to outliers since it squares errors, which may cause instability when predicting bounding boxes.  

---

### b. Cross-Entropy Loss (Classification Loss)
For predicting object classes, **Cross-Entropy Loss** is the most commonly used loss function in classification tasks. It is defined as:  

$$
\mathcal{L}_{CE} = -\sum_{i=1}^{C} y_i \log(\hat{y}_i)
$$  

where $ C $ is the total number of object classes, $ y_i $ is the ground truth label (one-hot encoded), and $ \hat{y}_i $ is the predicted probability for that class.  

- **Pros:** Standard for classification, trains the classifier from its logits.  
- **Cons:** Can be affected by class imbalances; label smoothing or weighted loss may be needed.  

---

### c. Huber Loss (Smooth Bounding Box Regression Loss)
Huber Loss is an improvement over MSE that reduces sensitivity to outliers. It applies MSE for small errors and Mean Absolute Error (MAE) for larger errors:  

$$
\mathcal{L}_{Huber} =
\begin{cases}
\frac{1}{2} (y_i - \hat{y}_i)^2, & \text{if} \ |y_i - \hat{y}_i| \leq \delta \\
\delta (|y_i - \hat{y}_i| - \frac{1}{2} \delta), & \text{otherwise}
\end{cases}
$$  

where $ \delta $ is a threshold that determines when the loss transitions from quadratic to linear.  

- **Pros:** More robust than MSE, reduces the effect of outliers on bounding box predictions.  
- **Cons:** Requires tuning of $ \delta $ for optimal performance.  

---

### d. Intersection over Union (IoU) Loss (Bounding Box Alignment Loss)
IoU Loss directly optimizes the overlap between the predicted and ground-truth bounding boxes:  

$$
\mathcal{L}_{IoU} = 1 - \frac{\text{Intersection Area}}{\text{Union Area}}
$$  

This loss ensures that the model prioritizes bounding box alignment rather than just minimizing coordinate differences.  

- **Pros:** More appropriate for object detection since it directly optimizes box overlap.  
- **Cons:** Harder to optimize, as gradients may vanish when boxes do not overlap.  

---

[PyTorch Documentation](https://pytorch.org/docs/stable/nn.html#loss-functions)

---

**Optimizers and their options:**

There are some pre-built [Optimizers in PyTorch](https://pytorch.org/docs/stable/optim.html), they are sufficient in most cases, especially if their parameters are well set. Two common choices are Adam and SGD; AdamW is a related optimizer with decoupled weight decay. They use gradients to update model parameters.

* **S**tochastic **G**radient **D**escent ([SGD](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html))
* **ADA**ptive **M**oment optimizer ([ADAM](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html))
* [A good general overview](https://www.ruder.io/optimizing-gradient-descent/)

The implemented composite uses cross-entropy and Smooth L1 with weights 1 and 5. The alternatives above provide context; the training code retains its original loss and Adam optimizer.


In [ ]:
class CompositeLoss(nn.Module):
    def __init__(self, lambda_cls=1.0, lambda_bbox=5.0):
        """
        Composite loss function combining:
        - CrossEntropyLoss for classification.
        - SmoothL1Loss (Huber Loss) for bounding box regression.

        lambda_cls: Weight for classification loss.
        lambda_bbox: Weight for bounding box loss.
        """
        super(CompositeLoss, self).__init__()
        self.classification_loss = nn.CrossEntropyLoss()  # Class probability loss
        self.bbox_loss = nn.SmoothL1Loss()  # Huber loss for bounding box regression
        self.lambda_cls = lambda_cls
        self.lambda_bbox = lambda_bbox

    def forward(self, pred_classes, true_classes, pred_bboxes, true_bboxes): # Computes the total loss.

        class_loss = self.classification_loss(pred_classes, true_classes)
        bbox_loss = self.bbox_loss(pred_bboxes, true_bboxes)

        total_loss = self.lambda_cls * class_loss + self.lambda_bbox * bbox_loss
        return total_loss, class_loss, bbox_loss

# Determine Loss function
loss_fn = CompositeLoss()

# Simulated batch data
batch_size = 10
num_classes = 3

#   Predictions
pred_classes = torch.randn(batch_size, num_classes)  # Logits for 3 classes
true_classes = torch.randint(0, num_classes, (batch_size,))  # True class labels

pred_bboxes = torch.randn(batch_size, 4)  # Predicted bounding boxes
true_bboxes = torch.randn(batch_size, 4)  # Ground truth bounding boxes

# Compute loss
total_loss, class_loss, bbox_loss = loss_fn(pred_classes, true_classes, pred_bboxes, true_bboxes)

print(f"Classification Loss: {class_loss.item():.4f}")
print(f"Bounding Box Loss: {bbox_loss.item():.4f}")
print(f"Total Loss: {total_loss.item():.4f}")


## 7. Training the Custom Object Detection Model

The training loop runs for up to 20 epochs. Each epoch records training and
validation loss. Adam updates the model at an initial learning rate of `1e-4`,
and the learning-rate scheduler reduces that rate when validation loss stalls.

The lowest validation loss selects the best checkpoint. Five epochs without an
improvement trigger early stopping. The test split is reserved for evaluation.

Every completed epoch saves model weights, Adam state, scheduler state, mixed-precision scaler, random states and history. Re-running the cell resumes this run. Optional mounted Drive preserves checkpoints between sessions; otherwise download the run archive before the remote runtime ends. With Drive enabled, training stops if mounting fails; checkpoints, metrics and figures are written to Drive. Mixed precision and asynchronous transfers reduce GPU overhead without replacing the original architecture.


In [ ]:
# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CustomObjectDetectionModel(num_classes=num_classes, fine_tune_backbone=True).to(device)
criterion = CompositeLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler('cuda',enabled=USE_AMP)
RUN_CONFIG = {'dataset_id':DATASET_ID,'model':f'resnet18-pool{POOL_GRID}',
              'pool_grid':POOL_GRID,
              'batch_size':train_loader.batch_size,'input_size':640,'seed':SEED,'amp':USE_AMP}

def save_checkpoint_atomic(value,path):
    temporary = path.with_suffix('.tmp')
    torch.save(value,temporary)
    temporary.replace(path)


#  Training Loop
def train_one_epoch(model, optimizer, train_loader, device, print_debug=False):
    model.train()
    total_loss = 0.0
    for batch_idx, (images, targets) in enumerate(tqdm(train_loader, desc="Training")):
        images = images.to(device,non_blocking=True)

        try:
            if print_debug and batch_idx == 0:
                print("--- First Batch Information ---")
                print("Targets type:", type(targets))
                if isinstance(targets, dict):
                    print("Targets keys:", targets.keys())
                    if 'labels' in targets:
                        print("Targets['labels'] shape:", targets['labels'].shape)
                    if 'boxes' in targets:
                        print("Targets['boxes'] shape:", targets['boxes'].shape)
                else:
                    print("Targets shape:", targets.shape)
                print("--- End First Batch Information ---")

            if isinstance(targets, dict) and 'labels' in targets and 'boxes' in targets:
                labels = targets['labels'].to(device).view(-1)
                boxes = targets['boxes'].to(device).view(-1, 4)
            elif isinstance(targets, list):
                labels = torch.cat([t['labels'].to(device) for t in targets]).view(-1)
                boxes = torch.cat([t['boxes'].to(device) for t in targets]).view(-1, 4)
            elif isinstance(targets, torch.Tensor):
                labels = targets[:, :, 0].long().to(device).view(-1)
                boxes = targets[:, :, 1:].to(device).view(-1, 4)
            else:
                raise ValueError(f"Unexpected target type: {type(targets)}")

        except (KeyError, TypeError, IndexError) as e:
            raise ValueError(f"Invalid training batch {batch_idx}") from e

        optimizer.zero_grad()
        with torch.amp.autocast('cuda',enabled=USE_AMP):
            class_preds, bbox_preds = model(images)
            loss, _, _ = criterion(class_preds, labels, bbox_preds, boxes)
        if not torch.isfinite(loss): raise FloatingPointError('Nonfinite training loss')
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(train_loader.dataset)

def validate_one_epoch(model, val_loader, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Validation"):
            images = images.to(device,non_blocking=True)
            try:
                if isinstance(targets, dict) and 'labels' in targets and 'boxes' in targets:
                    labels = targets['labels'].to(device).view(-1)
                    boxes = targets['boxes'].to(device).view(-1, 4)
                elif isinstance(targets, list):
                    labels = torch.cat([t['labels'].to(device) for t in targets]).view(-1)
                    boxes = torch.cat([t['boxes'].to(device) for t in targets]).view(-1, 4)
                elif isinstance(targets, torch.Tensor):
                    labels = targets[:, :, 0].long().to(device).view(-1)
                    boxes = targets[:, :, 1:].to(device).view(-1, 4)
                else:
                    raise ValueError(f"Unexpected target type: {type(targets)}")
            except (KeyError, TypeError, IndexError) as e:
                raise ValueError("Invalid validation batch") from e
            with torch.amp.autocast('cuda',enabled=USE_AMP):
                class_preds, bbox_preds = model(images)
                loss, _, _ = criterion(class_preds, labels, bbox_preds, boxes)
            if not torch.isfinite(loss): raise FloatingPointError('Nonfinite validation loss')
            total_loss += loss.item() * images.size(0)
    return total_loss / len(val_loader.dataset)

#  Training
num_epochs = 20
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0

train_losses = []  # To store training losses
val_losses = []  # To store validation losses
results = []  # To store results for each epoch
results_dir = str(METRIC_DIR)  # Directory to save results
results_path = os.path.join(results_dir, "training_results.json")  # File to save results

# Create the results directory if it doesn't exist
os.makedirs(results_dir, exist_ok=True)

#  Learning Rate Scheduler Setup
# Setup the learning rate scheduler for reducing the learning rate when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)

start_epoch = 0
if TRAIN_CUSTOM and BEST_PATH.exists() and not LAST_PATH.exists():
    raise FileNotFoundError('Restore last_checkpoint.pth to resume this run, or choose a new model directory.')
if LAST_PATH.exists():
    # Only load checkpoints produced by this notebook or another trusted source.
    checkpoint = torch.load(LAST_PATH,map_location='cpu',weights_only=False)
    if checkpoint['config'] != RUN_CONFIG:
        raise ValueError('Checkpoint data/model/settings differ; choose a new checkpoint folder.')
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler'])
    scaler.load_state_dict(checkpoint['scaler'])
    start_epoch = checkpoint['epoch']+1
    results = checkpoint['history']
    train_losses = [r['train_loss'] for r in results]
    val_losses = [r['val_loss'] for r in results]
    best_val_loss = checkpoint['best_val_loss']
    epochs_no_improve = checkpoint['epochs_no_improve']
    torch.set_rng_state(checkpoint['torch_rng'])
    random.setstate(checkpoint['python_rng'])
    np.random.set_state(checkpoint['numpy_rng'])
    if device.type == 'cuda': torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
    if not BEST_PATH.exists(): raise FileNotFoundError('Restore best_model.pth together with last_checkpoint.pth.')
    print(f'Resuming after epoch {start_epoch}.')
elif not TRAIN_CUSTOM:
    if not BEST_PATH.exists():
        raise FileNotFoundError('Evaluation-only mode requires best_model.pth.')
    if Path(results_path).exists():
        results = json.loads(Path(results_path).read_text())
        train_losses = [r['train_loss'] for r in results]
        val_losses = [r['val_loss'] for r in results]

for epoch in range(start_epoch, num_epochs) if TRAIN_CUSTOM else []:
    if epochs_no_improve >= patience:
        print('This run already reached early stopping.')
        break
    # Training phase
    train_loss = train_one_epoch(model, optimizer, train_loader, device, print_debug=(epoch == 0))

    # Validation phase
    val_loss = validate_one_epoch(model, val_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    epoch_results = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
    }
    results.append(epoch_results)

    print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    # Save results periodically (e.g., every 5 epochs)
    if epoch % 5 == 0 or epoch == num_epochs - 1:  # Save at the end too
        with open(results_path, "w") as f:
            json.dump(results, f, indent=4)  # Save with indent for readability
        print(f"Training results saved to {results_path}")

    # Check if validation loss improved
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        save_checkpoint_atomic(model.state_dict(),BEST_PATH)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping criterion reached; saving this epoch before stopping.")

    # Step the scheduler to adjust learning rate based on validation loss
    scheduler.step(val_loss)  # Original learning-rate schedule.
    save_checkpoint_atomic({'config':RUN_CONFIG,'epoch':epoch,'model':model.state_dict(),
        'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
        'scaler':scaler.state_dict(),'history':results,'best_val_loss':best_val_loss,
        'epochs_no_improve':epochs_no_improve,'torch_rng':torch.get_rng_state(),
        'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),
        'cuda_rng':torch.cuda.get_rng_state_all() if device.type == 'cuda' else None},LAST_PATH)
    with open(results_path,'w') as file: json.dump(results,file,indent=2)

# Save final results
with open(results_path, "w") as f:
    json.dump(results, f, indent=4)
print(f"Final training results saved to {results_path}")



## 8.1 Visualizing Training Metrics

The selected checkpoint is loaded before evaluation. Training and validation
curves show how the combined classification and localization loss changes over
epochs. The same history is exported to CSV for later inspection.

In [ ]:
import csv  # ✅ Import CSV module

# Load the Best Model ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CustomObjectDetectionModel(num_classes=3).to(device)

if os.path.exists(BEST_PATH):
    model.load_state_dict(torch.load(BEST_PATH, map_location=device, weights_only=True))
    model.eval()  # Set to eval mode
    print("✅ Best model loaded for visualization.")
else:
    raise FileNotFoundError(f"Train the original model first: {BEST_PATH}")

# Load Loss Data
results_dir = str(METRIC_DIR)
results_path = os.path.join(results_dir, "training_results.json")

if os.path.exists(results_path):
    with open(results_path, "r") as f:
        loaded_results = json.load(f)
        train_losses = [item["train_loss"] for item in loaded_results]
        val_losses = [item["val_loss"] for item in loaded_results]
    print(f"✅ Loss data loaded from {results_path}")
else:
    print(f"⚠️ Warning: '{results_path}' not found. Loss visualization will not work.")
    train_losses, val_losses = [], []

# Define Function to Plot & Save CSV ---
def plot_losses_and_save_csv():
    if train_losses and val_losses:
        # Plot losses
        plt.figure(figsize=(10, 5))
        plt.plot(train_losses, label='Training Loss', marker='o')
        plt.plot(val_losses, label='Validation Loss', marker='s')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.title('Training and Validation Loss Curves')
        plt.legend()
        plt.grid(True)
        save_figure('loss-curves.png')
        plt.show()

        # Save loss data to CSV
        csv_path = os.path.join(results_dir, 'training_metrics.csv')
        with open(csv_path, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['Epoch', 'Training Loss', 'Validation Loss'])
            for epoch, (train_loss, val_loss) in enumerate(zip(train_losses, val_losses)):
                writer.writerow([epoch + 1, train_loss, val_loss])

        print(f"✅ Training metrics saved to {csv_path}")
    else:
        print("⚠️ No loss data available. Skipping plot and CSV save.")

#  Run the Function
plot_losses_and_save_csv()


## Analysis of Model Convergence

Read the training and validation curves together. Falling training loss indicates
that the model is fitting the generated examples. Validation loss shows whether
that improvement carries over to held-out composites.

A plateau suggests diminishing improvement under the current settings. A widening
gap between the curves can indicate overfitting. The checkpoint selected by the
lowest validation loss, rather than simply the final epoch, is used for inference.

Conclusions about the current run should follow the curves and saved history above.

## 8.2 Running Inference on the Custom Object Detection Model

The evaluation cell loads the trained checkpoint, passes test images through the
detector and compares predicted classes and boxes with their labels. A detection
is correct when its class matches and its box passes the IoU threshold.
Precision, recall, F1, mean IoU, mAP@0.5 and inference time summarize the
run.

In [ ]:
# 1. Dataset and model
#
# Both are defined once earlier in the notebook and reused here. This cell
# used to carry its own copies. When the neck gained POOL_GRID, the copy here
# kept 1 x 1 pooling, so loading the trained weights raised a size mismatch
# on the head. One definition removes that class of failure.

# 3. Data Loading Utilities
def custom_collate_fn(batch):
    """Handles variable numbers of bounding boxes per image"""
    images = [item[0] for item in batch]
    targets = [item[1] for item in batch]

    # Stack images
    images = torch.stack(images, dim=0)

    # Pad targets to have same number of boxes
    max_boxes = max(t.shape[0] for t in targets)
    padded_targets = []

    for t in targets:
        padding = torch.zeros((max_boxes - t.shape[0], 5), dtype=t.dtype)
        padded_targets.append(torch.cat([t, padding], dim=0))

    targets = torch.stack(padded_targets, dim=0)
    return images, targets

#  4. Evaluation Metrics
def calculate_iou(pred_box, true_box):
    """Calculates Intersection over Union for normalized boxes"""
    pred_box = pred_box.clone()
    true_box = true_box.clone()

    if not torch.isfinite(pred_box).all() or (pred_box[2:] <= 0).any():
        return pred_box.new_tensor(0.0)
    # Convert center coordinates to corners
    pred_x1 = pred_box[0] - pred_box[2] / 2
    pred_y1 = pred_box[1] - pred_box[3] / 2
    pred_x2 = pred_box[0] + pred_box[2] / 2
    pred_y2 = pred_box[1] + pred_box[3] / 2

    true_x1 = true_box[0] - true_box[2] / 2
    true_y1 = true_box[1] - true_box[3] / 2
    true_x2 = true_box[0] + true_box[2] / 2
    true_y2 = true_box[1] + true_box[3] / 2

    # Calculate intersection area
    inter_x1 = max(pred_x1, true_x1)
    inter_y1 = max(pred_y1, true_y1)
    inter_x2 = min(pred_x2, true_x2)
    inter_y2 = min(pred_y2, true_y2)

    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    # Calculate union area
    pred_area = (pred_x2 - pred_x1) * (pred_y2 - pred_y1)
    true_area = (true_x2 - true_x1) * (true_y2 - true_y1)
    union_area = pred_area + true_area - inter_area

    return inter_area / union_area.clamp(min=torch.finfo(pred_box.dtype).eps)

def calculate_map(records, num_classes):
    """Mean average precision at the IoU threshold used to build `records`.

    The threshold is applied upstream in `evaluate_batch`, which sets the
    `is true positive` flag. This function takes no threshold of its own: an
    argument here would look configurable while changing nothing.

    On protocol: this integrates the precision-recall curve directly after
    making precision monotonic. Ultralytics uses 101-point interpolation, so
    this score and YOLO's mAP50 are not interchangeable even at the same IoU
    threshold. They are reported separately for that reason.

    `records` holds one entry per prediction as
    (predicted class, confidence, is true positive, true class). Within each
    class the predictions are ranked by confidence, precision and recall are
    accumulated down that ranking, and average precision is the area under the
    resulting curve. mAP averages that over the classes present in the labels.

    Precision and recall at a fixed threshold say nothing about whether the
    model's confidence is ordered. mAP does: a false positive ranked above a
    true positive scores lower than the same pair in the opposite order.
    """
    average_precisions = []

    for class_id in range(num_classes):
        class_records = [r for r in records if r[0] == class_id]
        num_ground_truths = sum(1 for r in records if r[3] == class_id)

        if num_ground_truths == 0:
            continue
        if not class_records:
            average_precisions.append(0.0)
            continue

        class_records.sort(key=lambda r: r[1], reverse=True)

        true_positives = 0
        false_positives = 0
        precisions, recalls = [], []
        for _, _, is_tp, _ in class_records:
            if is_tp:
                true_positives += 1
            else:
                false_positives += 1
            precisions.append(true_positives / (true_positives + false_positives))
            recalls.append(true_positives / num_ground_truths)

        # Make precision monotonic so a dip at one rank cannot depress the
        # ranks above it, then integrate over recall.
        for i in range(len(precisions) - 2, -1, -1):
            precisions[i] = max(precisions[i], precisions[i + 1])

        area = 0.0
        previous_recall = 0.0
        for precision, recall in zip(precisions, recalls):
            area += precision * (recall - previous_recall)
            previous_recall = recall
        average_precisions.append(area)

    return sum(average_precisions) / len(average_precisions) if average_precisions else 0.0


def evaluate_batch(pred_boxes, true_boxes, pred_classes, true_classes,
                   pred_scores=None, iou_thresh=0.5):
    """Evaluates a single batch of predictions"""
    batch_metrics = {
        'true_positives': 0,
        'total_preds': 0,
        'total_trues': 0,
        'ious': [],
        'records': []
    }

    for i in range(true_boxes.shape[0]):  # Loop through batch
        # Padding has zero area; class 0 is a real target.
        valid_mask = (true_boxes[i][:,2] > 0) & (true_boxes[i][:,3] > 0)
        current_true_boxes = true_boxes[i][valid_mask]
        current_true_classes = true_classes[i][valid_mask]

        # Count ground truths
        batch_metrics['total_trues'] += len(current_true_boxes)

        # One prediction per image: the task places exactly one object in each scene.
        pred_box = pred_boxes[i]
        pred_class = pred_classes[i]
        batch_metrics['total_preds'] += 1

        # Find best matching true box
        best_iou = 0
        for j in range(len(current_true_boxes)):
            if pred_class == current_true_classes[j]:
                iou = calculate_iou(pred_box, current_true_boxes[j])
                if iou > best_iou:
                    best_iou = iou

        if best_iou >= iou_thresh:
            batch_metrics['true_positives'] += 1

        # Mean IoU is reported as a localization measure, so it is taken against
        # the label regardless of the predicted class. best_iou above is
        # class-conditional and would read 0 for a well-placed box under the
        # wrong label, mixing the two heads into one number.
        localization_iou = 0.0
        for j in range(len(current_true_boxes)):
            overlap = float(calculate_iou(pred_box, current_true_boxes[j]))
            localization_iou = max(localization_iou, overlap)
        batch_metrics['ious'].append(localization_iou)

        confidence = float(pred_scores[i]) if pred_scores is not None else 1.0
        true_class = int(current_true_classes[0]) if len(current_true_classes) else -1
        batch_metrics['records'].append(
            (int(pred_class), confidence, bool(best_iou >= iou_thresh), true_class)
        )

    return batch_metrics

#  5. Testing Function
def test_model(model, test_loader, device, num_classes):
    model.eval()
    total_metrics = {
        'true_positives': 0,
        'total_preds': 0,
        'total_trues': 0
    }
    all_ious = []
    all_records = []
    inference_times = []

    with torch.no_grad():
        for images, targets in tqdm(test_loader, desc="Testing"):
            images = images.to(device)
            targets = targets.to(device)

            # Get true boxes and classes
            true_boxes = targets[..., 1:]  # Shape: [batch_size, max_boxes, 4]
            true_classes = targets[..., 0].long()  # Shape: [batch_size, max_boxes]

            # Measure inference time
            if device.type == 'cuda': torch.cuda.synchronize()
            start_time = time.perf_counter()
            class_logits, bbox_preds = model(images)
            if device.type == 'cuda': torch.cuda.synchronize()
            inference_times.append(time.perf_counter() - start_time)

            # Process predictions
            pred_classes = torch.argmax(class_logits, dim=1)  # Shape: [batch_size]
            pred_scores = torch.softmax(class_logits, dim=1).max(dim=1).values
            pred_boxes = bbox_preds  # Shape: [batch_size, 4]

            # Evaluate batch
            batch_metrics = evaluate_batch(
                pred_boxes, true_boxes,
                pred_classes, true_classes,
                pred_scores=pred_scores
            )

            # Accumulate metrics
            for k in total_metrics:
                total_metrics[k] += batch_metrics[k]
            all_ious.extend(batch_metrics['ious'])
            all_records.extend(batch_metrics['records'])

    # Calculate final metrics
    precision = total_metrics['true_positives'] / (total_metrics['total_preds'] + 1e-6)
    recall = total_metrics['true_positives'] / (total_metrics['total_trues'] + 1e-6)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-6)

    mean_iou = float(sum(all_ious) / len(all_ious)) if all_ious else 0.0
    mean_ap = calculate_map(all_records, num_classes)

    return {
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'mean_iou': mean_iou,
        'mAP@0.5': float(mean_ap),
        'avg_inference_time_ms': float(sum(inference_times) * 1000 / len(test_loader.dataset)),
        'num_test_samples': len(test_loader.dataset),
        'timing_scope': 'synchronized forward time per image; includes first batch, excludes data loading'
    }

#  6. Main Execution
if __name__ == '__main__':
    # Configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    num_classes = 3
    batch_size = 8
    model_path = str(BEST_PATH)
    backbone_type = "resnet18"  # or "vgg16"

    # Create test dataset
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    test_dataset = ObjectDetectionDataset(
        root_dir='dataset',
        split='test',
        num_classes=num_classes,
        transform=test_transform
    )

    # Create test loader
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=custom_collate_fn
    )

    # Initialize model
    model = CustomObjectDetectionModel(
        num_classes=num_classes,
        backbone_type=backbone_type,
        # Training fine-tuned the backbone, so the evaluated model is built the
        # same way. Predictions are unaffected either way under no_grad; this
        # keeps any parameter count describing the model that was trained.
        fine_tune_backbone=True
    ).to(device)

    # Load trained weights
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
        print(f"Loaded model weights from {model_path}")
    else:
        raise FileNotFoundError(f"Train the original model first: {model_path}")

    # Run evaluation
    print("\nStarting evaluation...")
    metrics = test_model(model, test_loader, device, num_classes)

    # Print results
    print("\nEvaluation Results:")
    print(f"{'Number of test samples:':<25} {metrics['num_test_samples']}")
    print(f"{'Precision:':<25} {metrics['precision']:.4f}")
    print(f"{'Recall:':<25} {metrics['recall']:.4f}")
    print(f"{'F1 Score:':<25} {metrics['f1']:.4f}")
    print(f"{'Mean IoU:':<25} {metrics['mean_iou']:.4f}")
    print(f"{'mAP@0.5:':<25} {metrics['mAP@0.5']:.4f}")
    print(f"{'Avg Inference Time:':<25} {metrics['avg_inference_time_ms']:.2f} ms")
with open(METRIC_DIR/'custom_test.json','w') as file:
    json.dump({'dataset_id':DATASET_ID,'metrics':metrics},file,indent=2)


## Explanation of Metrics

- **Test samples:** the number of held-out composites evaluated.
- **Precision:** correct detections divided by all predicted detections.
- **Recall:** correct detections divided by all ground-truth objects.
- **F1:** the harmonic mean of precision and recall.
- **IoU:** the intersection of two boxes divided by their union; it determines
  whether localization is accurate enough to count as a correct detection.
- **Mean IoU:** the average overlap between each predicted box and its label,
  taken regardless of the predicted class. Precision and recall fold the two
  heads into one number, so a well-placed box under the wrong label scores the
  same as a box in the wrong place. Mean IoU isolates the regression head.
- **mAP@0.5:** average precision at an IoU threshold of 0.5, computed here by
  integrating the precision-recall curve directly. Ultralytics uses 101-point
  interpolation for YOLO's `mAP50`, so the two numbers follow different
  protocols and are reported separately rather than compared directly. Per class
  by ranking predictions on confidence and integrating the precision-recall
  curve, then averaged over the classes present. Unlike precision at a fixed
  threshold, it rewards a model whose confidence is ordered: a false positive
  ranked above a true positive scores lower than the same pair reversed.
- **Inference time:** measured model execution time, interpreted together with
  the device, batch size and timing scope.

This custom model predicts one box for every image containing one target. Its
precision and recall should therefore agree. Use the current evaluation output;
historical scores are not evidence for code or data that has changed.

## 8.3 Visualizing Model Predictions

The original prediction layout places test scenes in a vertical sequence.
Ground-truth boxes are green and predictions are red dashed boxes. The annotation
shows the predicted class, true class, overlap and whether the detection meets
the correctness criterion. This connects aggregate metrics to individual successes
and localization errors.

In [ ]:
# Set up matplotlib for better visualization
plt.style.use('ggplot')
%matplotlib inline

def visualize_predictions(model, test_dataset, device, num_samples=5, iou_threshold=0.5):
    """
    Visualizes model predictions with ground truth boxes and metrics

    Args:
        model: Trained model
        test_dataset: Test dataset object
        device: Device (cpu/cuda)
        num_samples: Number of samples to visualize
        iou_threshold: IoU threshold for correct detection
    """
    model.eval()

    # Select random samples (ensure we don't request more than available)
    num_samples = min(num_samples, len(test_dataset))
    sample_indices = random.sample(range(len(test_dataset)), num_samples)

    # Set up figure
    fig, axes = plt.subplots(num_samples, 1, figsize=(15, 4*num_samples))
    if num_samples == 1:
        axes = [axes]  # Ensure axes is always a list

    for i, idx in enumerate(sample_indices):
        # Get sample and prepare for visualization
        image_tensor, target = test_dataset[idx]
        image_np = image_tensor.numpy().transpose(1, 2, 0)  # CHW to HWC
        image_np = (image_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
        image_np = np.clip(image_np, 0, 1)

        # Get ground truth (filter boxes)
        true_boxes = target[(target[:,3] > 0) & (target[:,4] > 0)]
        true_classes = true_boxes[:, 0].long()
        true_boxes = true_boxes[:, 1:]

        # Get prediction
        with torch.no_grad():
            input_tensor = image_tensor.unsqueeze(0).to(device)
            class_logits, bbox_pred = model(input_tensor)
            pred_class = torch.argmax(class_logits).item()
            pred_box = bbox_pred.squeeze().cpu().numpy()

        # Calculate metrics
        ious = []
        for box in true_boxes:
            iou = calculate_iou(torch.tensor(pred_box), box)
            ious.append(iou.item())
        max_iou = max(ious) if ious else 0
        is_correct = (pred_class in true_classes.numpy()) and (max_iou >= iou_threshold)

        # Create visualization
        ax = axes[i]
        ax.imshow(image_np)
        ax.axis('off')

        # Plot ground truth boxes (green)
        for box, class_id in zip(true_boxes, true_classes):
            x, y, w, h = box.numpy()
            rect = patches.Rectangle(
                ((x-w/2)*image_np.shape[1], (y-h/2)*image_np.shape[0]),
                w*image_np.shape[1],
                h*image_np.shape[0],
                linewidth=2, edgecolor='lime', facecolor='none', alpha=0.7
            )
            ax.add_patch(rect)
            ax.text((x-w/2)*image_np.shape[1], (y-h/2)*image_np.shape[0]-5,
                    f'GT: {class_id.item()}', color='lime', fontsize=10,
                    bbox=dict(facecolor='black', alpha=0.5, pad=1))

        # Plot predicted box (red)
        x, y, w, h = pred_box
        rect = patches.Rectangle(
            ((x-w/2)*image_np.shape[1], (y-h/2)*image_np.shape[0]),
            w*image_np.shape[1],
            h*image_np.shape[0],
            linewidth=2, edgecolor='red', facecolor='none', linestyle='--', alpha=0.7
        )
        ax.add_patch(rect)

        # Add detailed annotation
        annotation_text = (
            f"Pred: {pred_class} | GT: {true_classes.tolist()}\n"
            f"Max IoU: {max_iou:.2f} | Correct: {'✓' if is_correct else '✗'}"
        )
        ax.text(0.02, 0.98, annotation_text,
                transform=ax.transAxes, color='white', fontsize=10,
                verticalalignment='top', bbox=dict(facecolor='black', alpha=0.7, pad=5))

    plt.tight_layout()
    save_figure('predictions.png')
    plt.show()

# --- Updated Main Execution ---
if __name__ == '__main__':
    # Configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    num_classes = 3
    batch_size = 8
    model_path = str(BEST_PATH)
    backbone_type = "resnet18"

    # Create test dataset and loader
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    test_dataset = ObjectDetectionDataset(
        root_dir='dataset',
        split='test',
        num_classes=num_classes,
        transform=test_transform
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=custom_collate_fn
    )

    # Initialize and load model
    model = CustomObjectDetectionModel(
        num_classes=num_classes,
        backbone_type=backbone_type,
        fine_tune_backbone=False
    ).to(device)

    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
        print(f"Loaded model weights from {model_path}")

    else:
        raise FileNotFoundError(model_path)

    # Run evaluation and visualization
    print("\nStarting evaluation...")
    metrics = test_model(model, test_loader, device, num_classes)

    print("\nEvaluation Results:")
    for k, v in metrics.items():
        print(f"{k:>20}: {v:.4f}" if isinstance(v, float) else f"{k:>20}: {v}")

    print("\nVisualizing predictions...")
    visualize_predictions(model, test_dataset, device,
                        num_samples=5)

with open(METRIC_DIR/'custom_test.json','w') as file:
    json.dump({'dataset_id':DATASET_ID,'metrics':metrics},file,indent=2)


## 9. Loading an Existing Object Detection Model (YOLOv8n)

YOLOv8n provides a pretrained detection reference. It is fine-tuned on the same
generated dataset for up to 100 epochs at 640-pixel resolution. The dataset YAML
points to the existing train, validation and test directories and names the three
classes. [Ultralytics documentation](https://docs.ultralytics.com/) describes its
training and evaluation interfaces.
The saved scenes contain one object each. YOLO retains its standard training augmentations, including mosaic, which can combine objects from several scenes. New runs use the dataloader batch size and mixed-precision setting above; resumed runs retain their saved training settings.


In [ ]:
# Release completed custom-model training state before allocating YOLO.
for _name in ['model','optimizer','scheduler','scaler','checkpoint']:
    globals().pop(_name,None)
gc.collect()
if device.type == 'cuda': torch.cuda.empty_cache()
from ultralytics import YOLO, settings
# Keep experiment artifacts local; disable the optional cloud logger.
settings.update({"wandb": False})


### Configure the shared dataset


In [ ]:
data_yaml = {
    'path':DATASET_DIR.resolve().as_posix(),
    'train':'background/train/images','val':'background/val/images','test':'background/test/images',
    'nc':3,'names':['Waldo','Wenda','Wizard Whitebeard']}
YOLO_DATA = WORK_DIR/'data.yaml'  # Runtime paths stay outside the immutable dataset.
YOLO_DATA.write_text(json.dumps(data_yaml,indent=2),encoding='utf-8')
print('Dataset configuration:',YOLO_DATA)


### Verify split sizes


In [ ]:
for split in ['train','val','test']:
    directory = DATASET_DIR/'background'/split/'images'
    print(f'{split}: {len(list(directory.glob("*.jpg")))} images')
assert verify_dataset() == DATASET_ID


### Train or resume YOLOv8n


In [ ]:
YOLO_PROJECT = (DRIVE_DIR/'yolo') if DRIVE_DIR else OUTPUT_DIR/'yolo'
YOLO_RUN = YOLO_PROJECT/'train'
YOLO_LAST = YOLO_RUN/'weights/last.pt'
YOLO_BEST = YOLO_RUN/'weights/best.pt'
YOLO_ID_FILE = YOLO_PROJECT/'dataset_id.txt'
YOLO_PROJECT.mkdir(parents=True,exist_ok=True)
if YOLO_ID_FILE.exists() and YOLO_ID_FILE.read_text().strip() != DATASET_ID:
    raise ValueError('YOLO checkpoint belongs to a different dataset.')
YOLO_ID_FILE.write_text(DATASET_ID+'\n')
if TRAIN_YOLO:
    if YOLO_LAST.exists():
        saved = torch.load(YOLO_LAST,map_location='cpu',weights_only=False)
        completed_epochs = saved.get('epoch',-1)+1
        requested_epochs = saved.get('train_args',{}).get('epochs',100)
        if 0 < completed_epochs < requested_epochs:
            resume_path = YOLO_LAST
            if saved['train_args'].get('data') != str(YOLO_DATA):
                # A derived copy preserves weights, optimizer and epoch; only the data path changes.
                saved['train_args'] = dict(saved['train_args'],data=str(YOLO_DATA))
                resume_path = YOLO_RUN/'weights/resume-current-path.pt'
                pending = resume_path.with_suffix('.tmp')
                torch.save(saved,pending)
                pending.replace(resume_path)
            del saved
            yolo_model = YOLO(str(resume_path))
            yolo_model.train(resume=True,data=str(YOLO_DATA),save_dir=str(YOLO_RUN),
                             workers=NUM_WORKERS,device=0 if device.type=='cuda' else 'cpu')
        elif not YOLO_BEST.exists():
            raise FileNotFoundError('Completed YOLO run has no best.pt checkpoint.')
        else:
            print('Completed YOLO run found; retaining its selected checkpoint.')
    else:
        if YOLO_RUN.exists() and any(YOLO_RUN.iterdir()):
            raise FileExistsError('Partial YOLO run without last.pt; inspect before starting a new run.')
        yolo_model = YOLO('yolov8n.pt')
        yolo_model.train(data=str(YOLO_DATA),epochs=100,imgsz=640,seed=SEED,
            project=str(YOLO_PROJECT),name='train',exist_ok=True,workers=NUM_WORKERS,
            batch=train_loader.batch_size,amp=USE_AMP,
            device=0 if device.type=='cuda' else 'cpu')
RESULTS_CSV = str(YOLO_RUN/'results.csv')


## 10. Evaluating the YOLO Model

The training log contains box, classification and distribution-focal losses,
together with validation precision, recall and mAP. The original two-panel figure
shows loss trends alongside metric trends.

Validation history describes training progress. Final model assessment uses the
selected checkpoint on the held-out test split, with that distinction made explicit.

### Reading the training log

Inspect the CSV columns before plotting the saved learning curves and metrics.

In [ ]:
import pandas as pd

RESULTS_CSV = str(YOLO_RUN/"results.csv")

df = pd.read_csv(RESULTS_CSV)
print(df.columns)


### Plot YOLO learning curves


In [ ]:
# Path to YOLO training results CSV
RESULTS_CSV = str(YOLO_RUN/"results.csv")  # Update this path if needed

# Ensure the file exists
if not os.path.exists(RESULTS_CSV):
    raise FileNotFoundError(f"Results file not found at {RESULTS_CSV}. Ensure training has completed successfully.")

# Load the training log data into a Pandas DataFrame
df = pd.read_csv(RESULTS_CSV)

# Create a figure with two subplots (Training Losses & Model Performance Metrics)
plt.figure(figsize=(12, 5))

# Plot Training Losses
plt.subplot(1, 2, 1)
plt.plot(df["epoch"], df["train/box_loss"], label="Box Loss", color='r')
plt.plot(df["epoch"], df["train/cls_loss"], label="Classification Loss", color='b')
plt.plot(df["epoch"], df["train/dfl_loss"], label="DFL Loss", color='g')  # Using "train/dfl_loss"
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Loss Trends")
plt.legend()
plt.grid()

# Plot Evaluation Metrics (mAP, Precision, Recall)
plt.subplot(1, 2, 2)
plt.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@50", color='purple')
plt.plot(df["epoch"], df["metrics/precision(B)"], label="Precision", color='orange')
plt.plot(df["epoch"], df["metrics/recall(B)"], label="Recall", color='pink')
plt.xlabel("Epochs")
plt.ylabel("Metric Score")
plt.title("Validation Performance Metrics")
plt.legend()
plt.grid()

# Adjust layout for better visualization
plt.tight_layout()

# Display the plots
save_figure('yolo-curves.png')
plt.show()


### Evaluate YOLO on the test set


In [ ]:
import pandas as pd

# Load results CSV
RESULTS_CSV = str(YOLO_RUN/"results.csv")
df = pd.read_csv(RESULTS_CSV)

# Get the final epoch metrics (last row)
final_metrics = df.iloc[-1][[
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)"
]]

# Print results
print("Final epoch validation metrics (not test results):")
print(final_metrics)

if not YOLO_BEST.exists(): raise FileNotFoundError(YOLO_BEST)
yolo_best = YOLO(str(YOLO_BEST))
yolo_test = yolo_best.val(data=str(YOLO_DATA),split='test',imgsz=640,
                         project=str(YOLO_PROJECT),name='test',workers=NUM_WORKERS)
with open(METRIC_DIR/'yolo_test.json','w') as file:
    json.dump({'dataset_id':DATASET_ID,'metrics':{k:float(v) for k,v in yolo_test.results_dict.items()}},file,indent=2)
print('Held-out YOLO test metrics:',yolo_test.results_dict)


### YOLO Model Performance Interpretation

Precision measures how often predicted detections are correct; recall measures
how many labeled objects are found. mAP50 summarizes precision-recall behavior
at IoU 0.5, while mAP50–95 averages across stricter overlap thresholds from 0.5
to 0.95.

Read validation trends and held-out test results separately. High scores on these
synthetic scenes do not establish performance on unseen artwork or real images.
The custom model and YOLO also have different output and evaluation behavior,
so their reported metrics need that context.

## Saving the Experiment

Keep the generated dataset, selected model weights, resume checkpoints and loss
history together with the executed notebook. Saved figures and metrics document
the run, while the fixed dataset makes local and Colab runs comparable.
The final export cell can also be run after custom training, before YOLO. It includes available checkpoints, metrics, figures and YOLO test artifacts. Software versions are recorded in `runtime.json`.


In [ ]:
export_run_artifacts()
print('Save the executed notebook too; its figures and output document the run.')
